# Do it yourself: Bayesian yes–no decisions with binary data

So far, you've been handed sliders on a machine someone else built. This one hands you the parts. You write the lines of code at the heart of the inference — the visualizations are already wired to whatever you return.

Remember step 2 (the test vs. the world)? The object is a test or classifier with **known sensitivity and specificity**, operating under an **unknown base rate (prevalence)**. We want to infer the **PPV** (and, by the same logic, the NPV), which depend on that unknown.

We are going to build the apparatus of Bayesian inference and decision step by step.

(This notebook is a companion exercise to this [https://dreavjr.github.io/inference-tutorial/](primer on statistical inference).)

### Setup

Running on Colab? The cell below installs everything this notebook needs. On a local Jupyter you can skip it if the packages are already present.

In [ ]:
!pip install -q numpy scipy matplotlib

## 1 · The data model

- We assume that sensitivity and specificity are known and fixed
- The base rate / prevalence is unknown
- We want to infer the PPV, the positive predictive value (same as precision)
- As data, we have a certain number of samples that were predicted as positive or negative. Only the predicted condition is known, we don't know the true labels

Also: $$PPV = \frac{\mathrm{sensitivity} \cdot \mathrm{prevalence}}{\mathrm{sensitivity} \cdot \mathrm{prevalence} + (1 - \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})}$$


In [ ]:
from functools import partial
from typing import Callable

import numpy as np
from scipy import stats, integrate
import matplotlib.pyplot as plt

### 1.1 · The generative model and the likelihood

We reason in terms of a process that could generate the data we observe — that leads us to the right form for the likelihood. The guiding question is: *"if I knew the prevalence, how likely would the predicted positive and negative counts I have be?"*

We assume the data are i.i.d.: the generative model draws each datum independently from the same distribution. So we can focus on a single datum. What's the chance for one datum to be labeled positive?

To be labeled positive, a datum has to be truly positive **and** correctly labeled, **or** truly negative **and** mislabeled. Call that whole "probability of being predicted positive" $\theta$:

$$\theta =  \mathrm{sensitivity} \cdot \mathrm{prevalence} + (1- \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})$$

Because sensitivity and specificity are fixed, $\theta$ is a strictly monotonic function of the prevalence (increasing whenever the test beats chance). The two therefore carry the *same* information: knowing $\theta$ is knowing the base rate — and, with it, anything we'd care to report, the PPV included. So we run the whole inference in the convenient variable $\theta$, where the data give a clean binomial likelihood, and convert only at the end.

How are the labels we observe generated given a value for $\theta$? Each sample is a Bernoulli trial with success probability $\theta$. So, the probability of observing a certain count of positive and negative labels is given by the [binomial distribution](https://en.wikipedia.org/wiki/Binomial_distribution), implemented in [scipy.stats.binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html).

In [ ]:
def likelihood(theta: float, *, num_positive_labels: int, num_negative_labels: int) -> float:
    # ⚠️ Do it yourself! Return the likelihood as a probability density for observing the labels
    #    assuming theta is known
    return ...

## 2 · The prior

The parameter $\theta$ is a proportion, thus contained in the interval $[0, 1]$. You can pick any prior that reflects how much you know about theta, e.g., an uniform prior. 

You should know that a binomial likelihood with a fixed number of trials has a [conjugate prior](https://en.wikipedia.org/wiki/Conjugate_prior), which makes computing the posterior trivial (and is also easy to interpret, its strength being measured as a number of pseudo-observations). The conjugate prior for the binomial is the [beta distribution](https://en.wikipedia.org/wiki/Beta_distribution), implemented in in [scipy.stats.beta](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html).

(Incidentally: the uniform distribution on the interval $[0, 1]$ is also a beta distribution!)

In [ ]:
def prior(theta: float) -> float:
    # ⚠️ Do it yourself! Encode your prior by returning the probability density at theta
    return ...

## 3 · Computing the unnormalized posterior

Bayes' theorem says the posterior is proportional to prior times likelihood:

$$p(\theta \mid \text{data}) \;\propto\; p(\theta)\,\cdot\,p(\text{data}\mid\theta)$$

Up to the constant that makes it integrate to one, that's the whole story — a **pointwise multiplication** of the two functions you just wrote. No grid, no loop: we keep it as a function of $\theta$ and multiply.

In [ ]:
def get_unnormalized_posterior(
        prior: Callable[[float], float],
        likelihood: Callable[[float], float]
) -> Callable[[float], float]:
    return lambda theta: prior(theta) * likelihood(theta)

## 4 · Normalizing the posterior

The missing normalization constant is $\int_0^1 p(\theta)\,p(\text{data}\mid\theta)\,\mathrm{d}\theta$, the value of the denominator of Bayes' theorem, sometimes called *evidence*. It rescales the curve so it integrates to one.

This step, apparently so simple, is *the* central challenge of Bayesian inference: in real models the integral is high-dimensional and has no closed form, which is the entire reason MCMC, variational inference, and their friends exist. Here $\theta$ is one-dimensional on $[0, 1]$, so plain numerical integration does the job.

(You don't always need the posterior normalized — to compare the *odds* of two hypotheses the constant cancels — but to read off probabilities and expectations, you do.)

In [ ]:
def normalize_posterior(raw_posterior: Callable[[float], float]) -> Callable[[float], float]:
    # ⚠️ Do it yourself! Integrate the unnormalized posterior over theta in [0, 1] to get
    #    the evidence — the denominator of Bayes' theorem.
    evidence: float = ...
    return lambda theta: raw_posterior(theta) / evidence

## 5 · From $\theta$ to the PPV

We now have the posterior for $\theta$, but we set out to learn the **PPV** — the probability that a *positive verdict* is actually correct. Luckily the PPV is a deterministic one-to-one function of $\theta$: each $\theta$ fixes one PPV (and vice-versa). Since $\theta$ is exactly $P(\text{predicted } +)$, substituting the prevalence away gives the PPV straight from $\theta$:

$$\mathrm{PPV}(\theta) \;=\; \frac{\mathrm{sensitivity}\,(\theta + \mathrm{specificity} - 1)}{(\mathrm{sensitivity} + \mathrm{specificity} - 1)\,\theta}.$$

One annoying but crucial technicality: the distribution of a continuous variable (such as $\theta$) ranges over *densities* not probabilities, so we can't just relabel the x-axis. Where the change from $\theta$ to PPV stretches the axis, the density has to compress (and vice-versa) so that the total probability stays at one. That local stretch factor is given by the [Jacobian](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant), which in our 1-dimensional case is simply the magnitude of the derivative of the transformation $\left|\mathrm{d}\theta/\mathrm{d}\,\mathrm{PPV}\right|$. It has a simple closed-form:

$$\left|\frac{\mathrm{d}\theta}{\mathrm{d}\,\mathrm{PPV}}\right| \;=\; \frac{(\mathrm{sensitivity}+\mathrm{specificity}-1)\,\theta^{2}}{\mathrm{sensitivity}\,(1-\mathrm{specificity})}.$$

The transformed density is then: $$p_{\mathrm{PPV}}(\mathrm{PPV}) = p_{\theta}\big(\theta(\mathrm{PPV})\big)\cdot\left|\mathrm{d}\theta/\mathrm{d}\,\mathrm{PPV}\right|.$$

In [ ]:
def ppv_from_theta(theta: float, *, sensitivity: float, specificity: float) -> float:
    return sensitivity * (theta + specificity - 1) / ((sensitivity + specificity - 1) * theta)


def theta_from_ppv(ppv: float, *, sensitivity: float, specificity: float) -> float:
    return sensitivity * (specificity - 1) / (ppv * (sensitivity + specificity - 1) - sensitivity)


def ppv_posterior_from_theta_posterior(
        theta_posterior: Callable[[float], float],
        *, sensitivity: float, specificity: float,
) -> Callable[[float], float]:
    def ppv_posterior(ppv):
        theta = theta_from_ppv(ppv, sensitivity=sensitivity, specificity=specificity)
        # |d theta / d PPV|: PPV(theta) is a rational function, so the Jacobian is closed-form.
        jacobian = (sensitivity + specificity - 1) * theta ** 2 / (sensitivity * (1 - specificity))
        return theta_posterior(theta) * jacobian
    # Re-normalize: PPV in [0, 1] maps to theta in [1 - spec, sens], so any theta-mass
    # outside that band (an impossible prevalence) is dropped and rescaled away.
    return normalize_posterior(ppv_posterior)

## 6 · Assembling and visualizing the whole inference

Now we wire the pieces together. Pick the known test characteristics and the observed counts, choose a prior, then run the chain: prior → likelihood → unnormalized posterior → normalized posterior → PPV posterior. Every function below is one you wrote (or were handed) above.

In [ ]:
# --- what we know, and what we observed ---
sensitivity, specificity = 0.90, 0.85
num_positive_labels, num_negative_labels = 14, 26      # the classifier's verdicts on 40 samples

# --- the inference, one line per stage ---
# A Beta(8, 2) prior: as if we had already seen 8 positives and 2 negatives — a fairly
# strong hunch that the predicted-positive rate is high. Swap in Beta(1, 1) for a flat prior
# and watch the posterior collapse onto the likelihood.
chosen_prior = partial(prior)
chosen_likelihood = partial(likelihood,
                            num_positive_labels=num_positive_labels,
                            num_negative_labels=num_negative_labels)

raw_posterior = get_unnormalized_posterior(chosen_prior, chosen_likelihood)
theta_posterior = normalize_posterior(raw_posterior)
ppv_posterior = ppv_posterior_from_theta_posterior(
    theta_posterior, sensitivity=sensitivity, specificity=specificity)

# Two point summaries. The PPV of the posterior-mean theta and the posterior-mean PPV differ
# slightly because the theta -> PPV map is curved (the posterior, not a point, is transformed).
theta_mean, _ = integrate.quad(lambda theta: theta * theta_posterior(theta), 0.0, 1.0)
mean_ppv, _ = integrate.quad(lambda ppv: ppv * ppv_posterior(ppv), 0.0, 1.0)
print(f"PPV at the posterior-mean theta = "
      f"{ppv_from_theta(theta_mean, sensitivity=sensitivity, specificity=specificity):.3f}")
print(f"posterior-mean PPV              = {mean_ppv:.3f}")

### Plotting the components

In [ ]:
# Each plot takes a density as a function float -> float and samples it on a grid itself.

def plot_curve(f, title, *, color="#C45A26", domain=(0.0, 1.0), xlabel=r"$\theta$", n=400):
    """A single density over its domain — used to eyeball the prior or the likelihood."""
    x = np.linspace(domain[0], domain[1], n)
    y = f(x)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(x, y, lw=2.4, color=color)
    ax.fill_between(x, y, color=color, alpha=0.10)
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel); ax.set_title(title)
    fig.tight_layout(); plt.show()


def plot_three(prior_f, like_f, post_f, *, domain=(0.0, 1.0), n=400):
    """The prior - likelihood - posterior triptych. Prior and likelihood are rescaled to the
    posterior's height so the three shapes can be compared on one axis."""
    x = np.linspace(domain[0], domain[1], n)
    post = post_f(x)
    peak = np.max(post)
    to_peak = lambda v: v / np.max(v) * peak if np.max(v) > 0 else v
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(x, to_peak(prior_f(x)), lw=2.2, color="#7C7563", label="prior (scaled)")
    ax.plot(x, to_peak(like_f(x)),  lw=2.2, color="#355E92", label="likelihood (scaled)")
    ax.plot(x, post, lw=3.0, color="#C45A26", label="posterior")
    ax.fill_between(x, post, color="#C45A26", alpha=0.10)
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(r"$\theta$  (predicted-positive rate)"); ax.set_ylabel("density")
    ax.set_title("prior  -  likelihood  -  posterior")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()


def plot_overlay(post_f, exact_f, *, domain=(0.0, 1.0), n=400):
    """Your numerical posterior against a closed-form reference — they should coincide."""
    x = np.linspace(domain[0], domain[1], n)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(x, post_f(x),  lw=4.0, color="#C45A26", alpha=0.55, label="numerical (your pipeline)")
    ax.plot(x, exact_f(x), lw=1.6, ls="--", color="#1A1A1A",
            label=r"analytic  beta($\alpha+k,\ \beta+n-k$)")
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(r"$\theta$"); ax.set_ylabel("density")
    ax.set_title("numerical posterior  vs  analytic beta")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()

In [ ]:
plot_three(chosen_prior, chosen_likelihood, theta_posterior)
plot_curve(ppv_posterior, "posterior for the PPV", color="#2E7D5B", xlabel="PPV")

## 7 · An easier way: conjugacy

The pipeline above works for *any* prior — that's its virtue. But the beta prior is the *conjugate prior* for the binomial likelihood: prior and posterior are both beta, so the update has a closed form, no need to integrate.

The intuition is, again, pseudo-counts. A $\mathrm{beta}(\alpha, \beta)$ prior behaves like having already seen $\alpha$ "positive" and $\beta$ "negative" pseudo-observations; its strength is $\alpha + \beta$, an equivalent number of samples. The data then add $k$ real positives and $n - k$ real negatives, and you simply add them up:

$$\text{beta}(\alpha, \beta)\ \xrightarrow{\;k\text{ pos},\ n-k\text{ neg}\;}\ \text{beta}(\alpha + k,\ \beta + n - k).$$

Let's compute that directly and overlay it on the numerical posterior. If the prior is a beta distribution and you set the hyperparameters $\alpha$ and $\beta$ right, they should sit exactly on top of each other. If they don't, the hyperparameters are mismatched, you didn't pick a beta prior, or... you got a small bug somewhere along the way.

In [ ]:
alpha = 1 # ⚠️ These hyperparameters correspond to a beta(1, 1)
beta = 1  #    Pick the ones corresponding to your chosen prior
analytic_posterior = stats.beta(alpha + num_positive_labels,   # alpha + k
                                beta + num_negative_labels)     # beta + (n - k)

plot_overlay(theta_posterior, analytic_posterior.pdf)

grid = np.linspace(0.001, 0.999, 2000)
max_gap = np.max(np.abs(theta_posterior(grid) - analytic_posterior.pdf(grid)))
print(f"largest gap between numerical and analytic posterior: {max_gap:.2e}")

## Further reading

* Kruschke, John K. (2014). **Doing Bayesian Data Analysis: A Tutorial with R, JAGS, and Stan** (2nd ed.). Academic Press.

  Book site: https://shop.elsevier.com/books/doing-bayesian-data-analysis/kruschke/978-0-12-405888-0

  The most accessible. Theoretical treatment is superficial but everything is explained in detail. Includes a cookbook for many practical scenarios.

* McElreath, Richard. (2020). **Statistical Rethinking: A Bayesian Course with Examples in R and Stan** (2nd ed.). Chapman & Hall/CRC.

  Book site: https://xcelab.net/rm/statistical-rethinking/

  GitHub: https://github.com/rmcelreath/rethinking

  Course materials: https://github.com/rmcelreath/stat_rethinking_2024

  YouTube channel: https://www.youtube.com/@rmcelreath

  Rigorous but accessible, great practical explanation that keeps its theoretical teeth. A great course based on this book is available on the web.

* Gelman, Andrew; Carlin, John B.; Stern, Hal S.; Dunson, David B.; Vehtari, Aki; and Rubin, Donald B. (2013). **Bayesian Data Analysis** (3rd ed.). CRC Press.

  Book site / free PDF: https://sites.stat.columbia.edu/gelman/book/

  Aki Vehtari — Bayesian Data Analysis course: https://avehtari.github.io/BDA_course_Aalto/

  *The* rigorous treatment for modern Bayesian inference, very comprehensive, post-graduate level.

- Gelman, Andrew; Vehtari, Aki; McElreath, Richard; Simpson, Daniel; Margossian, Charles C.; Yao, Yuling; Kennedy, Lauren; Gabry, Jonah; Bürkner, Paul-Christian; Modrák, Martin; and Leos Barajas, Vianey. (2026). **Bayesian Workflow**. Chapman & Hall/CRC.

Book site: https://avehtari.github.io/Bayesian-Workflow/

A companion to Bayesian Data Analysis / Statistical Rethinking focused on the practical workflow of Bayesian inference: iterative model building, checking, diagnostics, validation, and communication.